# Confirmatory sensitivity analyses for the HCFIM EEG study

## tl;dr

This notebook evaluates the main unresolved robustness questions from peer review in the same 88-participant cohort.

- Benjamini–Hochberg correction retained **6 of 775** coordinate tests for control versus AD, but **0 of 775** for control versus FTD.
- The fixed combined EEG pipeline achieved ROC AUC **0.814** for AD and **0.757** for FTD; 999-label-permutation Monte Carlo p-values were **0.001** and **0.005**.
- Feature-selection stability was moderate for AD (mean fold-pair Jaccard **0.561**) and low for FTD (**0.235**).
- The best low-dimensional subsets were **C+FIM** for AD (AUC **0.740**) and **FIM** for FTD (AUC **0.710**); the three-coordinate embedding was not uniformly superior.
- Fixed-duration τ/order results varied materially. For AD, first-60-epoch AUC ranged from **0.651** to **0.754**; for FTD it ranged from **0.628** to **0.757**.
- MMSE-only models reached AUC 1.0 because controls all had MMSE 30; this is diagnostic circularity, not evidence that EEG generalizes.

These analyses strengthen the audit trail but do not provide external validation or support diagnostic deployment.

## Context & Methods

This notebook is a companion to notebook 05. It tests whether the principal findings are robust to reasonable analytical alternatives while preserving subject-level separation and fold-local preprocessing.

### Key assumptions

- The released subject matrix and participant index are the controlling sources for classification and covariates.
- Coordinate-wise FDR tests are exploratory univariate inference, not a multivariate test of the full geometry.
- MMSE is clinically entangled with diagnosis and is constant at 30 in controls. MMSE-only and MMSE-augmented models are therefore leakage diagnostics, not deployable EEG models.
- The permutation test uses a fixed, prespecified Extra Trees pipeline with fold-local feature selection. It tests that fixed pipeline, not the entire prior model-search process.
- Fixed-duration and τ/order sensitivities use the first 60 ten-second epochs for every participant, the maximum duration shared by the full cohort.
- The alternative FIM ordering uses a Steinhaus–Johnson–Trotter path through ordinal patterns, so adjacent states differ by one adjacent transposition. It is a principled sensitivity choice, not a uniquely correct ordering.


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import platform
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from scipy import signal, stats
from scipy.io import loadmat
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 17536
N_PERMUTATIONS = 999
N_FIXED_EPOCHS = 60

def resolve_project_root() -> Path:
    candidates = []
    if os.environ.get("HCFIM_PROJECT_ROOT"):
        candidates.append(Path(os.environ["HCFIM_PROJECT_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "results" / "subject_features_full.csv").exists():
            return candidate
    raise FileNotFoundError("Run from the repository notebooks directory or set HCFIM_PROJECT_ROOT.")

PROJECT_ROOT = resolve_project_root()
RESULTS = PROJECT_ROOT / "results"
SRC = PROJECT_ROOT / "src"
OUT = RESULTS / "confirmatory_sensitivity"
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import itm_geometry as geom

def resolve_dataset_root(project_root: Path) -> Path:
    """Locate ds004504 derivatives or use HCFIM_DATASET_ROOT."""
    candidates = []
    env_root = os.environ.get("HCFIM_DATASET_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        project_root / "data" / "ds004504",
        project_root / "data" / "data-08-00095",
        project_root / "data",
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "participants.tsv").is_file() and (candidate / "derivatives").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ds004504. Set HCFIM_DATASET_ROOT to the dataset directory "
        "containing participants.tsv and derivatives/."
    )

DATASET_ROOT = resolve_dataset_root(PROJECT_ROOT)

plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.dpi": 120, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})
COLORS = {"H": "#214d7d", "C": "#d58a2a", "FIM": "#6f8f3d", "reference": "#777777"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Python {platform.python_version()} | pandas {pd.__version__}")
print(f"Outputs: {OUT}")

## Data

### 1. Load and reconcile the released matrices

Participant identifiers, diagnoses, age, and MMSE must match one-to-one before any adjusted model is fitted.

In [ ]:
subject_features = pd.read_csv(RESULTS / "subject_features_full.csv")
participants = pd.read_csv(RESULTS / "participants_index.csv")

analysis_frame = subject_features.merge(
    participants[["participant_id", "diagnosis", "age", "MMSE", "set_path", "set_exists"]],
    on=["participant_id", "diagnosis"], how="left", validate="one_to_one",
)
assert len(analysis_frame) == 88
assert analysis_frame[["age", "MMSE"]].notna().all().all()
assert analysis_frame["set_exists"].all()
assert analysis_frame["participant_id"].nunique() == 88

cohort = analysis_frame.groupby("diagnosis").agg(
    n=("participant_id", "size"), age_mean=("age", "mean"), age_sd=("age", "std"),
    mmse_mean=("MMSE", "mean"), mmse_sd=("MMSE", "std"),
).reset_index()
display(cohort.style.format({"age_mean": "{:.1f}", "age_sd": "{:.1f}", "mmse_mean": "{:.1f}", "mmse_sd": "{:.1f}"}))
print(f"Subject matrix: {subject_features.shape[0]} × {subject_features.shape[1]}")

## Results

### 2. Coordinate-wise multiplicity control

For every unique HCFIM mean coordinate, the two diagnostic groups are compared with a two-sided Mann–Whitney U test. H is invariant to disequilibrium distance, so duplicate Euclidean, Jensen–Shannon, and Wootters H columns are tested once per entropy definition. C remains entropy- and distance-specific. FIM depends only on the probability vector and its state ordering; consequently, FIM is deduplicated both across disequilibrium distances and across Shannon, Rényi, and Tsallis labels when they share the same ordinal probability domain, embedding dimension, delay, and ordering. The family therefore contains 25 region-band combinations × [7 H definitions + (7 × 3) C definitions + 3 unique ordinal FIM domains] = 775 tests per diagnostic comparison. Hedges' g is retained as a signed standardized effect-size aid.

In [ ]:
COMPARISONS = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order] * len(p_values) / np.arange(1, len(p_values) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted = np.empty_like(ranked)
    adjusted[order] = np.clip(ranked, 0, 1)
    return adjusted

def hedges_g(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, float), np.asarray(b, float)
    pooled_num = (len(a) - 1) * np.var(a, ddof=1) + (len(b) - 1) * np.var(b, ddof=1)
    pooled = np.sqrt(pooled_num / (len(a) + len(b) - 2))
    if not np.isfinite(pooled) or pooled == 0:
        return np.nan
    d = (np.mean(b) - np.mean(a)) / pooled
    correction = 1 - 3 / (4 * (len(a) + len(b)) - 9)
    return correction * d

def ordinal_probability_domain_key(transform: str) -> str:
    """Collapse entropy variants that use the same ordinal P and state order."""
    match = re.search(r"_m(\d+)_t(\d+)$", transform)
    if transform.startswith("ordinal_") and match:
        return f"ordinal_m{match.group(1)}_t{match.group(2)}"
    return transform

hcfim_mean_columns = sorted(
    [column for column in subject_features if column.startswith("hcfim__") and column.endswith("__mean")],
    key=lambda column: (0 if column.split("__")[1].startswith("ordinal_shannon") else 1, column),
)
unique_test_columns = []
seen_tests = set()
for column in hcfim_mean_columns:
    parts = column.split("__")
    transform, distance, region, band, coordinate = parts[1:6]
    distance_key = distance if coordinate == "C" else "distance_invariant"
    probability_domain = ordinal_probability_domain_key(transform) if coordinate == "FIM" else transform
    test_key = (probability_domain, distance_key, region, band, coordinate)
    if test_key in seen_tests:
        continue
    seen_tests.add(test_key)
    unique_test_columns.append((column, distance_key, probability_domain))

# 25 region-band combinations x (7 entropy coordinates + 21 distance-specific
# complexity coordinates + 3 unique ordinal-probability FIM domains) = 775.
assert len(unique_test_columns) == 775
fdr_rows = []
for comparison, groups in COMPARISONS.items():
    subset = subject_features[subject_features["diagnosis"].isin(groups)]
    for column, distance_key, probability_domain in unique_test_columns:
        a = subset.loc[subset["diagnosis"] == groups[0], column].dropna().to_numpy(float)
        b = subset.loc[subset["diagnosis"] == groups[1], column].dropna().to_numpy(float)
        statistic, p_value = stats.mannwhitneyu(a, b, alternative="two-sided")
        parts = column.split("__")
        fdr_rows.append({
            "comparison": comparison, "column": column,
            "transform": parts[1], "probability_domain": probability_domain,
            "distance": distance_key, "region": parts[3],
            "band": parts[4], "coordinate": parts[5],
            "n_control": len(a), "n_pathology": len(b),
            "control_mean": np.mean(a), "pathology_mean": np.mean(b),
            "hedges_g": hedges_g(a, b), "u_statistic": statistic, "p_value": p_value,
        })

fdr_results = pd.DataFrame(fdr_rows)
fdr_results["q_value"] = fdr_results.groupby("comparison")["p_value"].transform(
    lambda values: benjamini_hochberg(values.to_numpy())
)
fdr_results["significant_fdr_05"] = fdr_results["q_value"] < 0.05
fdr_results.to_csv(OUT / "coordinate_tests_fdr.csv", index=False)

fdr_summary = fdr_results.groupby("comparison").agg(
    tests=("column", "size"), discoveries_fdr_05=("significant_fdr_05", "sum"),
    minimum_p=("p_value", "min"), minimum_q=("q_value", "min"),
).reset_index()
fdr_summary.to_csv(OUT / "coordinate_tests_fdr_summary.csv", index=False)
display(fdr_summary.style.format({"minimum_p": "{:.3g}", "minimum_q": "{:.3g}"}))


In [ ]:
top_fdr = (
    fdr_results.sort_values(["comparison", "q_value", "p_value"])
    .groupby("comparison", as_index=False).head(8).copy()
)
top_fdr["analysis_label"] = np.where(
    top_fdr["coordinate"].eq("FIM"), top_fdr["probability_domain"], top_fdr["transform"]
)
top_fdr["label"] = (top_fdr["region"] + " · " + top_fdr["band"] + " · " + top_fdr["coordinate"]
                    + " · " + top_fdr["distance"] + "\n" + top_fdr["analysis_label"])
top_fdr["minus_log10_q"] = -np.log10(top_fdr["q_value"].clip(lower=1e-16))

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, (comparison, group) in zip(axes, top_fdr.groupby("comparison", sort=False)):
    ordered = group.sort_values("minus_log10_q")
    colors = [COLORS.get(coord, "#777777") for coord in ordered["coordinate"]]
    ax.barh(ordered["label"], ordered["minus_log10_q"], color=colors)
    ax.axvline(-np.log10(0.05), color="#555555", linestyle="--", linewidth=1, label="q = 0.05")
    ax.set_xlabel("−log10(BH q-value)")
    ax.set_title(comparison.replace("_", " ").title())
    ax.grid(axis="x", alpha=0.2)
    ax.legend(frameon=False)
fig.suptitle("Strongest coordinate-wise group differences after multiplicity control", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "fdr_coordinate_tests.png", bbox_inches="tight", facecolor="white")
plt.show()


### 3. Shared nested-CV evaluator

All subsequent classification checks use the same five outer and three inner folds. Imputation, scaling, feature selection, and model fitting occur inside the training data for each split.

In [ ]:
def nested_extra_trees(
    frame: pd.DataFrame,
    groups: list[str],
    feature_columns: list[str],
    *,
    label: str,
    n_estimators: int = 300,
) -> tuple[dict, pd.DataFrame, list[set[str]]]:
    subset = frame.loc[frame["diagnosis"].isin(groups)].copy()
    X = subset[feature_columns].to_numpy(float)
    y = (subset["diagnosis"] == groups[1]).astype(int).to_numpy()
    probabilities = np.full(len(y), np.nan)
    predictions = np.full(len(y), -1)
    selections = []
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(30, len(feature_columns)))),
            ("model", ExtraTreesClassifier(
                n_estimators=n_estimators, class_weight="balanced", random_state=SEED, n_jobs=-1,
            )),
        ])
        inner = StratifiedKFold(3, shuffle=True, random_state=SEED + fold)
        search = GridSearchCV(
            pipeline, {"model__max_depth": [2, 3, None]}, scoring="roc_auc", cv=inner, n_jobs=1,
        )
        search.fit(X[train_idx], y[train_idx])
        best = search.best_estimator_
        probabilities[test_idx] = best.predict_proba(X[test_idx])[:, 1]
        predictions[test_idx] = best.predict(X[test_idx])
        support = best.named_steps["select"].get_support()
        selections.append(set(np.asarray(feature_columns)[support]))
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    metrics = {
        "label": label, "groups": " | ".join(groups), "n_subjects": len(y),
        "n_features": len(feature_columns), "roc_auc": roc_auc_score(y, probabilities),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "f1": f1_score(y, predictions), "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    oof = pd.DataFrame({
        "participant_id": subset["participant_id"].to_numpy(), "diagnosis": subset["diagnosis"].to_numpy(),
        "y_true": y, "y_prob": probabilities, "y_pred": predictions,
    })
    return metrics, oof, selections

def pairwise_jaccard(sets: list[set[str]]) -> float:
    values = []
    for left, right in itertools.combinations(sets, 2):
        values.append(len(left & right) / len(left | right) if left | right else np.nan)
    return float(np.nanmean(values))

### 4. Prespecified H/C/FIM dimensional ablations

The two manuscript-highlighted regional geometries are evaluated with every nonempty subset of H, C, and FIM. This tests whether the three-coordinate representation adds predictive information beyond one- or two-coordinate alternatives.

In [ ]:
selected_geometries = [
    ("AD temporal alpha", ["control", "alzheimer"], "temporal", "alpha"),
    ("FTD temporal broadband", ["control", "frontotemporal_dementia"], "temporal", "broadband"),
]

ablation_rows = []
for geometry_label, groups, region, band in selected_geometries:
    base = f"hcfim__ordinal_shannon_m4_t1__euclidean__{region}__{band}"
    coordinate_columns = {coord: f"{base}__{coord}__mean" for coord in ["H", "C", "FIM"]}
    for size in [1, 2, 3]:
        for coordinates in itertools.combinations(["H", "C", "FIM"], size):
            label = "+".join(coordinates)
            metrics, _, _ = nested_extra_trees(
                subject_features, groups, [coordinate_columns[c] for c in coordinates],
                label=label,
            )
            metrics.update({"geometry": geometry_label, "coordinates": label, "dimension": size})
            ablation_rows.append(metrics)

ablation_results = pd.DataFrame(ablation_rows)
ablation_results.to_csv(OUT / "hcfim_dimensional_ablation.csv", index=False)
display(ablation_results[["geometry", "coordinates", "dimension", "roc_auc", "balanced_accuracy", "f1"]].style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, (geometry, group) in zip(axes, ablation_results.groupby("geometry", sort=False)):
    ordered = group.sort_values("roc_auc", ascending=True)
    ax.barh(ordered["coordinates"], ordered["roc_auc"], color=["#214d7d" if d == 3 else "#69b3c2" if d == 2 else "#d58a2a" for d in ordered["dimension"]])
    ax.axvline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.set_xlim(0.25, 0.9)
    ax.set_xlabel("OOF ROC AUC")
    ax.set_title(geometry)
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Nested-CV H/C/FIM dimensional ablation")
fig.tight_layout()
fig.savefig(FIGURES / "hcfim_dimensional_ablation.png", bbox_inches="tight", facecolor="white")
plt.show()

### 5. Age/MMSE covariate diagnostics and feature-selection stability

The regional PSD+HCFIM matrix is compared with age-only, MMSE-only, age+MMSE-only, EEG+age, and EEG+age+MMSE variants. Covariates are forced through a separate fold-local preprocessing branch. Because MMSE is part of clinical characterization and perfectly separates controls in this dataset, any MMSE-associated gain must not be interpreted as EEG validation.

In [ ]:
def nested_covariate_model(
    frame: pd.DataFrame,
    groups: list[str],
    eeg_columns: list[str],
    covariate_columns: list[str],
    label: str,
) -> tuple[dict, pd.DataFrame, list[set[str]]]:
    subset = frame.loc[frame["diagnosis"].isin(groups)].copy()
    model_columns = eeg_columns + covariate_columns
    X = subset[model_columns]
    y = (subset["diagnosis"] == groups[1]).astype(int).to_numpy()
    transformers = []
    if eeg_columns:
        eeg_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(30, len(eeg_columns)))),
        ])
        transformers.append(("eeg", eeg_pipe, eeg_columns))
    if covariate_columns:
        cov_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        transformers.append(("covariates", cov_pipe, covariate_columns))
    preprocess = ColumnTransformer(transformers, remainder="drop")
    probabilities = np.full(len(y), np.nan)
    predictions = np.full(len(y), -1)
    selections = []
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
        pipeline = Pipeline([
            ("preprocess", preprocess),
            ("model", ExtraTreesClassifier(
                n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1,
            )),
        ])
        inner = StratifiedKFold(3, shuffle=True, random_state=SEED + fold)
        search = GridSearchCV(
            pipeline, {"model__max_depth": [2, 3, None]}, scoring="roc_auc", cv=inner, n_jobs=1,
        )
        search.fit(X.iloc[train_idx], y[train_idx])
        best = search.best_estimator_
        probabilities[test_idx] = best.predict_proba(X.iloc[test_idx])[:, 1]
        predictions[test_idx] = best.predict(X.iloc[test_idx])
        if eeg_columns:
            selector = best.named_steps["preprocess"].named_transformers_["eeg"].named_steps["select"]
            selections.append(set(np.asarray(eeg_columns)[selector.get_support()]))
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    metrics = {
        "label": label, "groups": " | ".join(groups), "n_subjects": len(y),
        "n_eeg_features": len(eeg_columns), "covariates": "+".join(covariate_columns) or "none",
        "roc_auc": roc_auc_score(y, probabilities),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "f1": f1_score(y, predictions), "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    oof = pd.DataFrame({
        "participant_id": subset["participant_id"].to_numpy(), "diagnosis": subset["diagnosis"].to_numpy(),
        "y_true": y, "y_prob": probabilities, "y_pred": predictions,
    })
    return metrics, oof, selections

eeg_columns = [column for column in subject_features if column.startswith("psd__") or column.startswith("hcfim__")]
covariate_variants = [
    ("EEG", eeg_columns, []),
    ("Age only", [], ["age"]),
    ("MMSE only", [], ["MMSE"]),
    ("Age + MMSE", [], ["age", "MMSE"]),
    ("EEG + age", eeg_columns, ["age"]),
    ("EEG + age + MMSE", eeg_columns, ["age", "MMSE"]),
]

covariate_rows, selection_records = [], []
for comparison, groups in COMPARISONS.items():
    for label, eeg_cols, cov_cols in covariate_variants:
        metrics, oof, selections = nested_covariate_model(analysis_frame, groups, eeg_cols, cov_cols, label)
        metrics["comparison"] = comparison
        covariate_rows.append(metrics)
        oof.to_csv(OUT / f"oof_covariate_{comparison}_{label.lower().replace(' ', '_').replace('+', 'plus')}.csv", index=False)
        if label == "EEG":
            for fold, selected in enumerate(selections, start=1):
                for feature in selected:
                    selection_records.append({"comparison": comparison, "fold": fold, "feature": feature})

covariate_results = pd.DataFrame(covariate_rows)
covariate_results.to_csv(OUT / "covariate_model_comparison.csv", index=False)
display(covariate_results[["comparison", "label", "roc_auc", "balanced_accuracy", "f1", "tn", "fp", "fn", "tp"]].style.format(precision=3))

selection_long = pd.DataFrame(selection_records)
selection_frequency = (
    selection_long.groupby(["comparison", "feature"])["fold"].nunique().rename("selected_folds").reset_index()
)
selection_frequency["selection_frequency"] = selection_frequency["selected_folds"] / 5
selection_frequency.to_csv(OUT / "feature_selection_frequency.csv", index=False)

stability_rows = []
for comparison in COMPARISONS:
    fold_sets = [set(group["feature"]) for _, group in selection_long[selection_long["comparison"] == comparison].groupby("fold")]
    stability_rows.append({"comparison": comparison, "mean_pairwise_jaccard": pairwise_jaccard(fold_sets)})
selection_stability = pd.DataFrame(stability_rows)
selection_stability.to_csv(OUT / "feature_selection_stability.csv", index=False)
display(selection_stability.style.format(precision=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True)
for ax, (comparison, group) in zip(axes, covariate_results.groupby("comparison", sort=False)):
    ordered = group.sort_values("roc_auc")
    colors = ["#d58a2a" if "MMSE" in label else "#214d7d" if label.startswith("EEG") else "#69b3c2" for label in ordered["label"]]
    bars = ax.barh(ordered["label"], ordered["roc_auc"], color=colors)
    ax.axvline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0.4, 1.03)
    ax.set_xlabel("OOF ROC AUC")
    ax.set_title(comparison.replace("_", " ").title())
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Covariate diagnostics: MMSE-enhanced performance is clinically circular")
fig.tight_layout()
fig.savefig(FIGURES / "covariate_model_comparison.png", bbox_inches="tight", facecolor="white")
plt.show()

top_stable = (
    selection_frequency.sort_values(["comparison", "selected_folds", "feature"], ascending=[True, False, True])
    .groupby("comparison", as_index=False).head(15)
)
display(top_stable)

### 6. Label-permutation sensitivity

For computational tractability and a clean null hypothesis, this test fixes the combined regional pipeline at 100 Extra Trees and `max_depth=3`, while retaining fold-local imputation, scaling, and selection of 30 features. Labels are permuted 999 times; the Monte Carlo p-value has minimum resolution 0.001. This is a conditional test of the fixed pipeline, not a correction for the earlier broad model search.

In [ ]:
def fixed_pipeline_oof_auc(X: np.ndarray, y: np.ndarray) -> float:
    probabilities = np.full(len(y), np.nan)
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for train_idx, test_idx in outer.split(X, y):
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(30, X.shape[1]))),
            ("model", ExtraTreesClassifier(
                n_estimators=100, max_depth=3, class_weight="balanced", random_state=SEED, n_jobs=1,
            )),
        ])
        pipeline.fit(X[train_idx], y[train_idx])
        probabilities[test_idx] = pipeline.predict_proba(X[test_idx])[:, 1]
    return roc_auc_score(y, probabilities)

permutation_rows, permutation_summary_rows = [], []
for comparison, groups in COMPARISONS.items():
    subset = subject_features[subject_features["diagnosis"].isin(groups)]
    columns = [column for column in subset if column.startswith("psd__") or column.startswith("hcfim__")]
    X = subset[columns].to_numpy(float)
    y = (subset["diagnosis"] == groups[1]).astype(int).to_numpy()
    observed_auc = fixed_pipeline_oof_auc(X, y)
    rng = np.random.default_rng(SEED)
    permuted_labels = [rng.permutation(y) for _ in range(N_PERMUTATIONS)]
    null_aucs = Parallel(n_jobs=-1, prefer="threads")(
        delayed(fixed_pipeline_oof_auc)(X, permuted_y) for permuted_y in permuted_labels
    )
    p_value = (1 + np.sum(np.asarray(null_aucs) >= observed_auc)) / (N_PERMUTATIONS + 1)
    permutation_summary_rows.append({
        "comparison": comparison, "observed_auc": observed_auc,
        "null_mean_auc": np.mean(null_aucs), "null_sd_auc": np.std(null_aucs, ddof=1),
        "n_permutations": N_PERMUTATIONS, "monte_carlo_p": p_value,
    })
    permutation_rows.extend({"comparison": comparison, "permutation": i + 1, "null_auc": value} for i, value in enumerate(null_aucs))

permutation_summary = pd.DataFrame(permutation_summary_rows)
permutation_distribution = pd.DataFrame(permutation_rows)
permutation_summary.to_csv(OUT / "label_permutation_summary.csv", index=False)
permutation_distribution.to_csv(OUT / "label_permutation_distribution.csv", index=False)
display(permutation_summary.style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), sharex=True)
for ax, row in zip(axes, permutation_summary.itertuples(index=False)):
    values = permutation_distribution.loc[permutation_distribution["comparison"] == row.comparison, "null_auc"]
    ax.hist(values, bins=15, color="#b7c9dc", edgecolor="white")
    ax.axvline(row.observed_auc, color="#214d7d", linewidth=2.2, label=f"Observed = {row.observed_auc:.3f}")
    ax.axvline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.set_title(row.comparison.replace("_", " ").title())
    ax.set_xlabel("OOF ROC AUC under label permutation")
    ax.set_ylabel("Permutations")
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "label_permutation_test.png", bbox_inches="tight", facecolor="white")
plt.show()

### 7. Fixed-duration, ordinal-delay, and FIM-ordering sensitivity

The raw EEGLAB matrices are read directly with SciPy. For each subject, the temporal regional signal is reconstructed exactly as in the released notebook and the first 60 epochs are filtered separately. Ordinal Shannon HCFIM is recomputed for `m=4`, τ = 1, 2, and 3. Lexicographic and Johnson–Trotter ordinal-state paths are compared for FIM at τ = 1.

In [ ]:
CHANNEL_RENAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
TEMPORAL_CHANNELS = ["T7", "T8", "P7", "P8"]
BANDS = {"alpha": (8.0, 13.0), "broadband": (0.5, 40.0)}

def sjt_permutations(n: int) -> list[tuple[int, ...]]:
    permutations = [(0,)]
    for value in range(1, n):
        expanded = []
        for index, permutation in enumerate(permutations):
            positions = range(value, -1, -1) if index % 2 == 0 else range(0, value + 1)
            for position in positions:
                candidate = list(permutation)
                candidate.insert(position, value)
                expanded.append(tuple(candidate))
        permutations = expanded
    return permutations

LEX_PATTERNS = list(itertools.permutations(range(4)))
LEX_INDEX = {pattern: index for index, pattern in enumerate(LEX_PATTERNS)}
SJT_INDEX = np.array([LEX_INDEX[pattern] for pattern in sjt_permutations(4)], dtype=int)
assert len(SJT_INDEX) == math.factorial(4)

def ordered_fim(probabilities: np.ndarray, indices: np.ndarray | None = None) -> float:
    p = probabilities if indices is None else probabilities[indices]
    return float(np.sum(np.diff(np.sqrt(p)) ** 2) / 2.0)

def load_set_matrix(set_path: Path) -> tuple[np.ndarray, float, list[str]]:
    payload = loadmat(set_path, simplify_cells=True)
    data = np.asarray(payload["data"], dtype=float)
    sfreq = float(payload["srate"])
    channel_locations = payload["chanlocs"]
    if isinstance(channel_locations, dict):
        channel_locations = [channel_locations]
    labels = [CHANNEL_RENAME.get(str(item["labels"]), str(item["labels"])) for item in channel_locations]
    return data, sfreq, labels

def epoch_slices(n_samples: int, sfreq: float, n_epochs: int = N_FIXED_EPOCHS):
    window = int(round(10.0 * sfreq))
    step = int(round(window * 0.5))
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        if epoch_id >= n_epochs:
            break
        yield epoch_id, start, start + window

def bandpass_window(values: np.ndarray, sfreq: float, band: tuple[float, float]) -> np.ndarray:
    values = signal.detrend(np.asarray(values, float), type="constant")
    nyquist = sfreq / 2.0
    low, high = band
    sos = signal.butter(4, [max(low, 0.01) / nyquist, min(high, nyquist - 0.5) / nyquist], btype="bandpass", output="sos")
    return signal.sosfiltfilt(sos, values)

def extract_selected_sensitivities(row) -> list[dict]:
    data, sfreq, labels = load_set_matrix(Path(row.set_path))
    channel_index = {label: index for index, label in enumerate(labels)}
    indices = [channel_index[channel] for channel in TEMPORAL_CHANNELS if channel in channel_index]
    if not indices:
        raise ValueError(f"No temporal channels for {row.participant_id}")
    regional = np.nanmean(data[indices, :], axis=0)
    accumulators = {}
    configs = [("AD temporal alpha", "alpha"), ("FTD temporal broadband", "broadband")]
    for _, start, end in epoch_slices(regional.size, sfreq):
        window = regional[start:end]
        for configuration, band_name in configs:
            filtered = bandpass_window(window, sfreq, BANDS[band_name])
            for tau in [1, 2, 3]:
                probabilities = geom.ordinal_probabilities(filtered, m=4, tau=tau)
                spec = geom.EntropySpec(
                    entropy_id=f"ordinal_shannon_m4_t{tau}", label=f"m4 tau{tau}", family="ordinal",
                    probability_domain="Bandt-Pompe ordinal patterns", n_states=24,
                    entropy_kind="shannon", m=4, tau=tau,
                )
                point = geom.hcfim_point(probabilities, spec, distance_code=geom.DISTANCES["euclidean"]["code"])
                variants = [(f"first60_tau{tau}_lex", point["FIM"])]
                if tau == 1:
                    variants.append(("first60_tau1_sjt", ordered_fim(probabilities, SJT_INDEX)))
                for variant, fim in variants:
                    key = (configuration, variant, tau, "sjt" if variant.endswith("sjt") else "lexicographic")
                    accumulator = accumulators.setdefault(key, {"H": [], "C": [], "FIM": []})
                    accumulator["H"].append(point["H"])
                    accumulator["C"].append(point["C"])
                    accumulator["FIM"].append(fim)
    output = []
    for (configuration, variant, tau, ordering), values in accumulators.items():
        output.append({
            "participant_id": row.participant_id, "diagnosis": row.diagnosis,
            "configuration": configuration, "variant": variant, "tau": tau, "ordering": ordering,
            "n_epochs": len(values["H"]),
            "H": np.mean(values["H"]), "C": np.mean(values["C"]), "FIM": np.mean(values["FIM"]),
        })
    return output

In [ ]:
raw_rows = []
for index, row in enumerate(analysis_frame.itertuples(index=False), start=1):
    if index == 1 or index % 10 == 0 or index == len(analysis_frame):
        print(f"[{index:02d}/{len(analysis_frame)}] {row.participant_id}")
    raw_rows.extend(extract_selected_sensitivities(row))

selected_sensitivities = pd.DataFrame(raw_rows)
assert selected_sensitivities["n_epochs"].eq(N_FIXED_EPOCHS).all()

# Add the released full-duration tau=1 lexicographic means for a like-for-like duration comparison.
full_duration_rows = []
for configuration, groups, region, band in selected_geometries:
    base = f"hcfim__ordinal_shannon_m4_t1__euclidean__{region}__{band}"
    columns = {coord: f"{base}__{coord}__mean" for coord in ["H", "C", "FIM"]}
    for row in subject_features.itertuples(index=False):
        record = {"participant_id": row.participant_id, "diagnosis": row.diagnosis, "configuration": configuration,
                  "variant": "full_duration_tau1_lex", "tau": 1, "ordering": "lexicographic", "n_epochs": np.nan}
        for coordinate, column in columns.items():
            record[coordinate] = getattr(row, column)
        full_duration_rows.append(record)

selected_sensitivities = pd.concat([selected_sensitivities, pd.DataFrame(full_duration_rows)], ignore_index=True)
selected_sensitivities.to_csv(OUT / "selected_geometry_sensitivity_features.csv", index=False)
display(selected_sensitivities.groupby(["configuration", "variant"])[["H", "C", "FIM"]].mean().round(4))

In [ ]:
sensitivity_rows = []
for configuration, groups, _, _ in selected_geometries:
    config_data = selected_sensitivities[selected_sensitivities["configuration"] == configuration]
    for variant in ["full_duration_tau1_lex", "first60_tau1_lex", "first60_tau2_lex", "first60_tau3_lex", "first60_tau1_sjt"]:
        variant_data = config_data[config_data["variant"] == variant][["participant_id", "diagnosis", "H", "C", "FIM"]]
        metrics, _, _ = nested_extra_trees(variant_data, groups, ["H", "C", "FIM"], label=variant)
        metrics.update({"configuration": configuration, "variant": variant})
        sensitivity_rows.append(metrics)

sensitivity_results = pd.DataFrame(sensitivity_rows)
sensitivity_results.to_csv(OUT / "duration_tau_ordering_sensitivity.csv", index=False)
display(sensitivity_results[["configuration", "variant", "roc_auc", "balanced_accuracy", "f1"]].style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharex=True)
label_map = {
    "full_duration_tau1_lex": "Full duration · τ1 · lex",
    "first60_tau1_lex": "First 60 · τ1 · lex",
    "first60_tau2_lex": "First 60 · τ2 · lex",
    "first60_tau3_lex": "First 60 · τ3 · lex",
    "first60_tau1_sjt": "First 60 · τ1 · SJT",
}
for ax, (configuration, group) in zip(axes, sensitivity_results.groupby("configuration", sort=False)):
    ordered = group.copy()
    ordered["display"] = ordered["variant"].map(label_map)
    ordered = ordered.sort_values("roc_auc")
    colors = ["#d58a2a" if "sjt" in variant else "#69b3c2" if "first60" in variant else "#214d7d" for variant in ordered["variant"]]
    bars = ax.barh(ordered["display"], ordered["roc_auc"], color=colors)
    ax.axvline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0.25, 0.95)
    ax.set_xlabel("OOF ROC AUC")
    ax.set_title(configuration)
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Fixed-duration, ordinal-delay, and FIM-ordering sensitivity")
fig.tight_layout()
fig.savefig(FIGURES / "duration_tau_ordering_sensitivity.png", bbox_inches="tight", facecolor="white")
plt.show()

## Takeaways

- Multiplicity control substantially narrowed the geometric evidence: AD retained 6 coordinate-wise discoveries, whereas FTD retained none.
- The fixed EEG pipeline performed above its permuted-label null distribution, but this conditional test does not correct the earlier broad representation/model search.
- Age alone did not materially improve discrimination. MMSE produced near-perfect or perfect separation because it is clinically entangled with diagnosis and constant in controls; MMSE-adjusted gains must not be attributed to EEG.
- Coordinate ablations and fold-selection stability show that the high-dimensional representation is not uniquely determined, especially for FTD.
- Duration matching, ordinal delay, and FIM ordering changed performance enough to affect interpretation. τ and ordering should be prespecified in future work rather than selected after viewing outcomes.
- All findings remain internal to one cohort. Independent external validation is still required.

In [ ]:
coverage = pd.DataFrame([
    {"reviewer_request": "Label permutation", "status": "completed", "scope": "999 permutations; fixed combined Extra Trees pipeline"},
    {"reviewer_request": "FDR correction", "status": "completed", "scope": "BH across 775 unique tests; H deduplicated across distance and FIM across distance plus shared ordinal probability domains"},
    {"reviewer_request": "Age/MMSE covariates", "status": "completed", "scope": "covariate-only and EEG-augmented nested-CV diagnostics"},
    {"reviewer_request": "Fixed duration", "status": "completed", "scope": "first 60 epochs in the two selected temporal geometries"},
    {"reviewer_request": "τ sensitivity", "status": "completed", "scope": "m=4, τ=1/2/3, first 60 epochs"},
    {"reviewer_request": "FIM ordering", "status": "completed", "scope": "lexicographic versus Johnson–Trotter, τ=1, first 60 epochs"},
    {"reviewer_request": "H/C/FIM ablation", "status": "completed", "scope": "all seven nonempty coordinate subsets in selected geometries"},
    {"reviewer_request": "Feature-selection stability", "status": "completed", "scope": "outer-fold SelectKBest frequency and pairwise Jaccard"},
    {"reviewer_request": "External validation", "status": "not available", "scope": "no independent cohort in repository"},
])
coverage.to_csv(OUT / "analysis_coverage.csv", index=False)
display(coverage)

manifest = {
    "project_root": str(PROJECT_ROOT), "dataset_root": str(DATASET_ROOT),
    "seed": SEED, "n_permutations": N_PERMUTATIONS, "fixed_epochs": N_FIXED_EPOCHS,
    "n_subjects": int(subject_features["participant_id"].nunique()),
    "outputs": sorted(str(path.relative_to(PROJECT_ROOT)) for path in OUT.rglob("*") if path.is_file()),
}
(OUT / "notebook_06_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Validation status: all executed cells completed and all internal assertions passed.")
print(json.dumps(manifest, indent=2))